# Classical Baseline: TF-IDF + LogisticRegression

Phase 5 deliverable. Trains on the same splits produced by `split_issues.py` and evaluated
against the same test set used by the DistilBERT fine-tune.

**Prerequisites**
- MinIO running locally (`docker compose up minio`)
- `.env` at repo root with `MINIO_ACCESS_KEY`, `MINIO_SECRET_KEY`, `MINIO_LOCAL_ENDPOINT`, `MINIO_BUCKET`
- Splits uploaded (`uv run python backend/scripts/split_issues.py`)

**Run from**: VS Code with the Jupyter extension, or `jupyter nbconvert --to notebook --execute`.

In [ ]:
# Cell 1 — imports and env
from __future__ import annotations

import hashlib
import io
import json
import os
import pickle
import sys
import time
import uuid
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from minio import Minio
from minio.error import S3Error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import Pipeline

REPO_ROOT = Path(".").resolve().parent  # notebooks/ -> repo root
load_dotenv(REPO_ROOT / ".env", override=False)

MINIO_ENDPOINT = os.environ["MINIO_LOCAL_ENDPOINT"].removeprefix("http://").removeprefix("https://")
MINIO_ACCESS   = os.environ["MINIO_ACCESS_KEY"]
MINIO_SECRET   = os.environ["MINIO_SECRET_KEY"]
BUCKET         = os.environ["MINIO_BUCKET"]
SPLIT_PREFIX   = "splits/v1"
LABELS         = ["bug", "feature", "docs", "question"]

secure = not (MINIO_ENDPOINT.startswith("localhost") or MINIO_ENDPOINT.startswith("127."))
minio  = Minio(MINIO_ENDPOINT, access_key=MINIO_ACCESS, secret_key=MINIO_SECRET, secure=secure)
print(f"Connected to MinIO at {MINIO_ENDPOINT}, bucket={BUCKET}")

In [ ]:
# Cell 2 — download splits
def _download_split(split_name: str) -> list[dict]:
    key = f"{SPLIT_PREFIX}/{split_name}.jsonl"
    resp = minio.get_object(BUCKET, key)
    return [json.loads(line) for line in resp.read().decode("utf-8").splitlines() if line.strip()]

train_rows = _download_split("train")
test_rows  = _download_split("test")
print(f"Train: {len(train_rows)}   Test: {len(test_rows)}")

X_train = [r["text"] for r in train_rows]
y_train = [r["label"] for r in train_rows]
X_test  = [r["text"] for r in test_rows]
y_test  = [r["label"] for r in test_rows]

In [ ]:
# Cell 3 — build pipeline (D-P5-01)
# TF-IDF hyperparameter choices:
#   ngram_range=(1,2): unigrams + bigrams capture two-word phrases (e.g. "index error", "missing value")
#   max_features=50_000: vocabulary cap keeps memory bounded; covers >99% of term frequency mass
#   sublinear_tf=True: log(1+tf) dampens high-frequency terms, standard for text classification
#   min_df=2: drops hapax legomena (single-occurrence terms) that add noise
# LR hyperparameter choices:
#   C=1.0: default regularisation; cross-validated in notebook below
#   class_weight='balanced': compensates for the severe question-class imbalance (4.2% of train)
#   solver='lbfgs', multi_class='multinomial': correct for 4-class softmax
#   max_iter=1000: enough for convergence on this vocabulary size

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=50_000,
        sublinear_tf=True,
        min_df=2,
    )),
    ("clf", LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight="balanced",
        solver="lbfgs",
        multi_class="multinomial",
        random_state=42,
    )),
])
print("Pipeline built.")

In [ ]:
# Cell 4 — train (target: < 60s on local CPU)
t_start = time.perf_counter()
pipeline.fit(X_train, y_train)
train_secs = time.perf_counter() - t_start
print(f"Training time: {train_secs:.1f}s")
assert train_secs < 60, f"Training exceeded 60s ({train_secs:.1f}s) — check hardware"

In [ ]:
# Cell 5 — evaluate on test split (same split as Phase 4 DistilBERT)
y_pred    = pipeline.predict(X_test)
accuracy  = accuracy_score(y_test, y_pred)
macro_f1  = f1_score(y_test, y_pred, average="macro", labels=LABELS, zero_division=0)
per_class = f1_score(y_test, y_pred, average=None, labels=LABELS, zero_division=0)
per_class_dict = dict(zip(LABELS, per_class.tolist()))

print(f"Test accuracy : {accuracy:.4f}")
print(f"Test macro-F1 : {macro_f1:.4f}")
for cls, score in per_class_dict.items():
    print(f"  {cls:<10} F1 = {score:.4f}")
print()
print(classification_report(y_test, y_pred, labels=LABELS, zero_division=0))

In [ ]:
# Cell 6 — latency measurement (single-sample, cold + warm)
# Methodology: first 5 calls are "cold" (TF-IDF transform of unseen document).
# We report warm p50/p95 over 200 single-sample calls — the operational metric
# for a real-time triage endpoint.

n_warmup  = 5
n_measure = 200
samples   = (X_test * (n_measure // len(X_test) + 1))[:n_measure + n_warmup]

# cold
cold_latencies: list[float] = []
for text in samples[:n_warmup]:
    t0 = time.perf_counter()
    pipeline.predict([text])
    cold_latencies.append((time.perf_counter() - t0) * 1000)

# warm
warm_latencies: list[float] = []
for text in samples[n_warmup:n_warmup + n_measure]:
    t0 = time.perf_counter()
    pipeline.predict([text])
    warm_latencies.append((time.perf_counter() - t0) * 1000)

cold_p50 = float(np.percentile(cold_latencies, 50))
warm_p50 = float(np.percentile(warm_latencies, 50))
warm_p95 = float(np.percentile(warm_latencies, 95))

print(f"Cold  p50 : {cold_p50:.3f} ms  (n={n_warmup})")
print(f"Warm  p50 : {warm_p50:.3f} ms  (n={n_measure})")
print(f"Warm  p95 : {warm_p95:.3f} ms")

In [ ]:
# Cell 7 — upload pipeline to MinIO
run_id = str(uuid.uuid4())
buf    = io.BytesIO()
pickle.dump(pipeline, buf)
payload = buf.getvalue()
sha256  = hashlib.sha256(payload).hexdigest()
key     = f"models/classical/{run_id}/pipeline.pkl"

minio.put_object(
    BUCKET, key, io.BytesIO(payload), length=len(payload),
    content_type="application/octet-stream",
)
print(f"Uploaded: {key}")
print(f"SHA-256 : {sha256}")
print(f"Size    : {len(payload) / 1024:.1f} KB")

In [ ]:
# Cell 8 — summary
print("="*60)
print("CLASSICAL BASELINE SUMMARY")
print("="*60)
print(f"Model        : TF-IDF(ngram=(1,2), max_f=50k) + LR(C=1.0, balanced)")
print(f"Hardware     : local CPU")
print(f"Train time   : {train_secs:.1f}s")
print(f"Accuracy     : {accuracy:.4f}")
print(f"Macro-F1     : {macro_f1:.4f}")
for cls, score in per_class_dict.items():
    print(f"  {cls:<10} F1 = {score:.4f}")
print(f"Latency p50  : {warm_p50:.3f} ms (warm)")
print(f"Latency p95  : {warm_p95:.3f} ms (warm)")
print(f"Cost/1k preds: $0.00 (local CPU, post-train)")
print(f"MinIO key    : {key}")
print(f"Run ID       : {run_id}")